# 17 — CrewAI Knowledge and a separate LanceDB retriever
Knowledge's built-in store is not LanceDB. This adapter indexes the same
fictional documents separately and returns source IDs for inspection.



**Advanced follow-up to lesson 5 (knowledge).**


## Problem

Built-in Knowledge does not expose LanceDB as its default store.


## Concept

A separate LanceDB adapter can index and retrieve the same source-labeled documents.

CrewAI Knowledge and this LanceDB table are separate retrieval systems. Both need embeddings and source IDs.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents   ◀ YOU ARE HERE
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
fictional files → Ollama embeddings → LanceDB; query → vector search → source IDs (separate from Knowledge)
```

documents → Ollama embeddings → LanceDB → source-labeled matches. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — List source documents before embedding

List source documents before embedding.


In [ ]:
import os
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.retrieval import index_documents, search_index
from workshop.tools import DOCUMENTS

print("Available source IDs:", DOCUMENTS)



## Index and query with local Ollama embeddings
Requires the configured embedding model and a running Ollama service.



## Step 2 — Build a LanceDB table and search with the local embedding model

Build a LanceDB table and search with the local embedding model.


In [ ]:
if os.getenv("WORKSHOP_RUN_LIVE_DEMOS") == "true":
    print("Indexed:", index_documents())
    print("Results:", search_index("private deployment and deletion guarantee"))
else:
    print("Set WORKSHOP_RUN_LIVE_DEMOS=true to build and query the LanceDB index.")


## Failure mode to know about

A vector hit may be irrelevant even when its source ID is valid. Similarity is not truth.


## Production implication

Version the embedding model and rebuild after source changes. Test relevance and deletion separately.


## Your turn

Compare a LanceDB hit with CrewAI Knowledge's answer.

**Checkpoint:** Search returns only real source IDs from the fictional documents.
